# crdt-merge v0.5.0 — A100 Validation
**The Protocol Release**: Wire format, compression, probabilistic CRDTs

Tests v0.5.0-specific features + full system sanity check on NVIDIA A100.

In [ ]:
!pip install crdt-merge==0.5.0 psutil -q

import crdt_merge, sys, os, time, json, tracemalloc, gc
print(f'crdt-merge version: {crdt_merge.__version__}')
print(f'Python: {sys.version.split()[0]}')

try:
    import torch
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('torch not available (CPU-only)')

import psutil
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'CPUs: {os.cpu_count()}')

## Utilities

In [ ]:
import numpy as np

results = []

def record(suite, test, value, unit, passed=True):
    results.append({'suite': suite, 'test': test, 'value': value, 'unit': unit, 'passed': passed})
    status = '' if passed else ''
    print(f'  {status} {test}: {value} {unit}')

def measure_memory(fn):
    gc.collect()
    tracemalloc.start()
    result = fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak / 1e6  # MB

def gen_records(n, id_start=0):
    return [{'id': i, 'value': int(np.random.randint(0, 1000)),
             'ts': float(i + np.random.random()),
             'name': f'item_{i}'} for i in range(id_start, id_start + n)]

print('Utilities loaded ')

## Suite 1: Wire Format
Tests `serialize()`, `deserialize()`, `peek_type()`, `wire_size()`, and batch operations.

In [ ]:
from crdt_merge.wire import serialize, deserialize, peek_type, wire_size
from crdt_merge.wire import serialize_batch, deserialize_batch
from crdt_merge import GCounter, PNCounter, LWWRegister, ORSet, LWWMap

print('=' * 70)
print('SUITE 1: WIRE FORMAT')
print('=' * 70)

SUITE = 'wire'

# --- GCounter ---
print('\n--- GCounter ---')
gc = GCounter()
gc.increment('node_a', 100)
gc.increment('node_b', 200)
data = serialize(gc)
record(SUITE, 'gcounter_serialize', len(data), 'bytes', len(data) > 0)
ptype = peek_type(data)
record(SUITE, 'gcounter_peek_type', ptype, '', ptype == 'g_counter')
gc2 = deserialize(data)
record(SUITE, 'gcounter_roundtrip', gc2.value, '', gc2.value == 300)
sz = wire_size(data)
record(SUITE, 'gcounter_wire_size', sz, '', isinstance(sz, dict))

# --- PNCounter ---
print('\n--- PNCounter ---')
pn = PNCounter()
pn.increment('a', 50)
pn.decrement('b', 20)
data = serialize(pn)
pn2 = deserialize(data)
record(SUITE, 'pncounter_roundtrip', pn2.value, '', pn2.value == 30)
record(SUITE, 'pncounter_peek', peek_type(data), '', peek_type(data) == 'pn_counter')

# --- LWWRegister ---
print('\n--- LWWRegister ---')
reg = LWWRegister('hello', timestamp=42.0)
data = serialize(reg)
reg2 = deserialize(data)
record(SUITE, 'lww_register_roundtrip', reg2.value, '', reg2.value == 'hello')
record(SUITE, 'lww_register_timestamp', reg2.timestamp, '', reg2.timestamp == 42.0)

# --- ORSet ---
print('\n--- ORSet ---')
s = ORSet()
s.add('apple'); s.add('banana'); s.add('cherry')
data = serialize(s)
s2 = deserialize(data)
record(SUITE, 'orset_roundtrip', len(s2.value), 'elements', s2.value == s.value)

# --- LWWMap ---
print('\n--- LWWMap ---')
m = LWWMap()
m.set('name', 'Alice', timestamp=1.0)
m.set('age', 30, timestamp=2.0)
data = serialize(m)
m2 = deserialize(data)
record(SUITE, 'lwwmap_roundtrip_name', m2.get('name'), '', m2.get('name') == 'Alice')
record(SUITE, 'lwwmap_roundtrip_age', m2.get('age'), '', m2.get('age') == 30)

# --- Batch ---
print('\n--- Batch Operations ---')
objects = [GCounter(), PNCounter(), LWWRegister('test')]
objects[0].increment('a', 5)
batch_data = serialize_batch(objects)
record(SUITE, 'batch_serialize', len(batch_data), 'bytes', len(batch_data) > 0)
restored = deserialize_batch(batch_data)
record(SUITE, 'batch_deserialize', len(restored), 'objects', len(restored) == 3)
record(SUITE, 'batch_gcounter_value', restored[0].value, '', restored[0].value == 5)

print(f'\nWire suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 2: Compression
Tests wire format compression and size reduction.

In [ ]:
from crdt_merge.wire import serialize, deserialize
from crdt_merge import GCounter, LWWMap

print('=' * 70)
print('SUITE 2: COMPRESSION')
print('=' * 70)

SUITE = 'compression'

# --- Compression on GCounter ---
print('\n--- GCounter Compression ---')
gc = GCounter()
for i in range(100):
    gc.increment(f'node_{i}', i * 100)
raw = serialize(gc, compress=False)
compressed = serialize(gc, compress=True)
ratio = len(compressed) / len(raw) if len(raw) > 0 else 1
record(SUITE, 'gcounter_raw_size', len(raw), 'bytes')
record(SUITE, 'gcounter_compressed_size', len(compressed), 'bytes')
record(SUITE, 'gcounter_ratio', round(ratio, 3), 'x', ratio < 1.0)
gc2 = deserialize(compressed)
record(SUITE, 'gcounter_compressed_roundtrip', gc2.value, '', gc2.value == gc.value)

# --- Compression on large LWWMap ---
print('\n--- LWWMap Compression (1K entries) ---')
m = LWWMap()
for i in range(1000):
    m.set(f'key_{i}', f'value_{i}' * 10, timestamp=float(i))
raw = serialize(m, compress=False)
compressed = serialize(m, compress=True)
ratio = len(compressed) / len(raw)
record(SUITE, 'lwwmap_raw_size', len(raw), 'bytes')
record(SUITE, 'lwwmap_compressed_size', len(compressed), 'bytes')
record(SUITE, 'lwwmap_ratio', round(ratio, 3), 'x', ratio < 1.0)
m2 = deserialize(compressed)
record(SUITE, 'lwwmap_compressed_roundtrip', m2.get('key_0'), '', m2.get('key_0') == 'value_0' * 10)

# --- Throughput ---
print('\n--- Serialization Throughput ---')
gc = GCounter()
for i in range(50):
    gc.increment(f'n{i}', i)

t0 = time.time()
for _ in range(1000):
    serialize(gc)
ser_ms = (time.time() - t0) * 1000
record(SUITE, 'serialize_1k_ops_ms', round(ser_ms, 1), 'ms')

data = serialize(gc)
t0 = time.time()
for _ in range(1000):
    deserialize(data)
deser_ms = (time.time() - t0) * 1000
record(SUITE, 'deserialize_1k_ops_ms', round(deser_ms, 1), 'ms')

print(f'\nCompression suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 3: Probabilistic CRDTs
Tests `MergeableHLL`, `MergeableBloom`, `MergeableCMS`.

In [ ]:
from crdt_merge.probabilistic import MergeableHLL, MergeableBloom, MergeableCMS

print('=' * 70)
print('SUITE 3: PROBABILISTIC CRDTS')
print('=' * 70)

SUITE = 'probabilistic'

# --- HyperLogLog ---
print('\n--- MergeableHLL ---')
hll1 = MergeableHLL()
hll2 = MergeableHLL()
for i in range(5000):
    hll1.add(f'user_{i}')
for i in range(3000, 8000):
    hll2.add(f'user_{i}')

hll3 = hll1.merge(hll2)
est = hll3.cardinality()
# True cardinality = 8000 (users 0-7999)
error_pct = abs(est - 8000) / 8000 * 100
record(SUITE, 'hll_cardinality', round(est, 0), 'estimated', error_pct < 5)
record(SUITE, 'hll_error_pct', round(error_pct, 2), '%', error_pct < 5)
record(SUITE, 'hll_standard_error', round(hll3.standard_error(), 4), '')
record(SUITE, 'hll_size_bytes', hll3.size_bytes(), 'bytes', hll3.size_bytes() > 0)

# HLL serialization roundtrip
d = hll3.to_dict()
hll4 = MergeableHLL.from_dict(d)
record(SUITE, 'hll_roundtrip', round(hll4.cardinality(), 0), '', abs(hll4.cardinality() - est) < 1)

# --- Bloom Filter ---
print('\n--- MergeableBloom ---')
bf1 = MergeableBloom(capacity=10000, fp_rate=0.01)
bf2 = MergeableBloom(capacity=10000, fp_rate=0.01)
for i in range(5000):
    bf1.add(f'item_{i}')
for i in range(3000, 8000):
    bf2.add(f'item_{i}')

bf3 = bf1.merge(bf2)

# Check membership
tp = sum(1 for i in range(8000) if bf3.contains(f'item_{i}'))
record(SUITE, 'bloom_true_positives', tp, '/8000', tp == 8000)  # zero false negatives

fp = sum(1 for i in range(8000, 9000) if bf3.contains(f'item_{i}'))
fp_rate = fp / 1000
record(SUITE, 'bloom_false_positive_rate', round(fp_rate, 4), '', fp_rate < 0.05)
record(SUITE, 'bloom_estimated_fp', round(bf3.estimated_fp_rate(), 4), '')
record(SUITE, 'bloom_size_bytes', bf3.size_bytes(), 'bytes', bf3.size_bytes() > 0)

# Bloom serialization
d = bf3.to_dict()
bf4 = MergeableBloom.from_dict(d)
record(SUITE, 'bloom_roundtrip', bf4.contains('item_0'), '', bf4.contains('item_0'))

# --- Count-Min Sketch ---
print('\n--- MergeableCMS ---')
cms1 = MergeableCMS()
cms2 = MergeableCMS()
for _ in range(100):
    cms1.add('popular')
for _ in range(50):
    cms2.add('popular')
cms1.add('rare')

cms3 = cms1.merge(cms2)
# CMS merge takes element-wise max per cell, so estimate = max(100, 50) = 100
record(SUITE, 'cms_popular_estimate', cms3.estimate('popular'), '', cms3.estimate('popular') >= 100)
record(SUITE, 'cms_rare_estimate', cms3.estimate('rare'), '', cms3.estimate('rare') >= 1)
record(SUITE, 'cms_total', cms3.total, '', cms3.total >= 100)
record(SUITE, 'cms_size_bytes', cms3.size_bytes(), 'bytes', cms3.size_bytes() > 0)

# CMS serialization
d = cms3.to_dict()
cms4 = MergeableCMS.from_dict(d)
record(SUITE, 'cms_roundtrip', cms4.estimate('popular'), '', cms4.estimate('popular') >= 100)

# --- Scale: HLL with 1M items ---
print('\n--- Scale: HLL 1M items ---')
hll_big = MergeableHLL()
t0 = time.time()
for i in range(1000000):
    hll_big.add(f'user_{i}')
add_ms = (time.time() - t0) * 1000
big_est = hll_big.cardinality()
big_err = abs(big_est - 1000000) / 1000000 * 100
record(SUITE, 'hll_1m_cardinality', round(big_est, 0), 'estimated', big_err < 5)
record(SUITE, 'hll_1m_error_pct', round(big_err, 2), '%', big_err < 5)
record(SUITE, 'hll_1m_add_time_ms', round(add_ms, 0), 'ms')
record(SUITE, 'hll_1m_size_bytes', hll_big.size_bytes(), 'bytes')

print(f'\nProbabilistic suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 4: System Sanity Check
Quick validation that all core features still work in v0.5.0.

In [ ]:
from crdt_merge import merge, diff, merge_with_provenance, export_provenance
from crdt_merge import MergeSchema, LWW, MaxWins
from crdt_merge import merge_sorted_stream, merge_stream, StreamStats
from crdt_merge import verified_merge
from crdt_merge.verify import verify_crdt
from crdt_merge.delta import compute_delta, apply_delta

print('=' * 70)
print('SUITE 4: SYSTEM SANITY')
print('=' * 70)

SUITE = 'sanity'

# --- Core merge ---
print('\n--- Core merge ---')
a = [{'id': 1, 'v': 'a', 'ts': 1.0}, {'id': 2, 'v': 'a', 'ts': 1.0}]
b = [{'id': 2, 'v': 'b', 'ts': 2.0}, {'id': 3, 'v': 'b', 'ts': 1.0}]
merged = merge(a, b, key='id', timestamp_col='ts')
record(SUITE, 'core_merge', len(merged), 'rows', len(merged) == 3)

# --- Provenance ---
print('\n--- Provenance ---')
m, prov = merge_with_provenance(a, b, key='id', timestamp_col='ts')
record(SUITE, 'provenance', prov.total_rows, 'rows', prov.total_rows == 3)

# --- Delta ---
print('\n--- Delta ---')
old = [{'id': 1, 'v': 'old'}]; new = [{'id': 1, 'v': 'new'}, {'id': 2, 'v': 'added'}]
delta = compute_delta(old, new, key='id')
applied = apply_delta(old, delta, key='id')
record(SUITE, 'delta', len(applied), 'rows', len(applied) == 2)

# --- Streaming ---
print('\n--- Streaming ---')
sa = gen_records(1000, 0); sb = gen_records(1000, 500)
stats = StreamStats()
batches = list(merge_sorted_stream(iter(sa), iter(sb), key='id', timestamp_col='ts', stats=stats))
total = sum(len(b) for b in batches)
record(SUITE, 'streaming', total, 'rows', total == 1500)

# --- Verify ---
print('\n--- Verify ---')
def gfn():
    n = np.random.randint(5, 15)
    ids = np.random.choice(range(30), size=n, replace=False).tolist()
    return [{'id': i, 'val': int(np.random.randint(100)), 'ts': float(np.random.random()*1000)} for i in ids]
def mfn(a, b): return merge(a, b, key='id', timestamp_col='ts')
def efn(a, b): return sorted(a, key=lambda r: r['id']) == sorted(b, key=lambda r: r['id'])
res = verify_crdt(mfn, gfn, trials=20, eq_fn=efn)
record(SUITE, 'verify_crdt', res.passed, '', res.passed)

# --- Scale: 50K merge ---
print('\n--- Scale: 50K merge ---')
a_big = gen_records(50000, 0); b_big = gen_records(50000, 25000)
t0 = time.time()
merged_big = merge(a_big, b_big, key='id', timestamp_col='ts')
elapsed = (time.time() - t0) * 1000
record(SUITE, 'scale_50k', len(merged_big), 'rows', len(merged_big) == 75000)
record(SUITE, 'scale_50k_ms', round(elapsed, 1), 'ms')

print(f'\nSanity suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Final Report

In [ ]:
print('=' * 70)
print('FINAL REPORT: crdt-merge v0.5.0 A100 Validation')
print('=' * 70)

total = len(results)
passed = sum(1 for r in results if r['passed'])
failed = total - passed

print(f'\nTotal measurements: {total}')
print(f'Passed: {passed}')
print(f'Failed: {failed}')

for suite in ['wire', 'compression', 'probabilistic', 'sanity']:
    suite_results = [r for r in results if r['suite'] == suite]
    sp = sum(1 for r in suite_results if r['passed'])
    st = len(suite_results)
    status = '' if sp == st else ''
    print(f'  {status} {suite}: {sp}/{st}')

if failed > 0:
    print('\n FAILURES:')
    for r in results:
        if not r['passed']:
            print(f'  - {r["suite"]}/{r["test"]}: {r["value"]} {r["unit"]}')

print(f'\n{" ALL TESTS PASSED" if failed == 0 else "️ SOME TESTS FAILED"}')

# Export results
with open('v050_a100_results.json', 'w') as f:
    json.dump({'version': '0.5.0', 'measurements': results,
               'total': total, 'passed': passed, 'failed': failed}, f, indent=2)
print(f'\nResults exported to v050_a100_results.json')